# YouTube Shorts Deduplication Test & Extrapolation

This notebook verifies our proposed algorithm for **YouTube Shorts deduplication**. 

### Objectives:
1. Group videos by channel and separate them into **Shorts** and **Long videos**.
2. Implement an optimized **word-level subset matching** algorithm that checks if an *entire Short transcript* is contained within a *subset of a Long video*.
3. Test the algorithm on a sample channel (`Camihawke`) and measure execution time.
4. Extrapolate the processing time to the entire dataset.

In [ ]:
import os
import time
import difflib
import pandas as pd

PARQUET_PATH = "Data_Cleaned/yt_videos_with_local_transcripts.parquet"
assert os.path.exists(PARQUET_PATH), f"Parquet file not found at {PARQUET_PATH}"

print("Loading dataset...")
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df):,} videos with {len(df.columns)} columns.")

Loading dataset...
Loaded 12,829 videos with 29 columns.


In [ ]:
# Filter dataset
df_shorts = df[df["is_short"] == True]
df_longs = df[df["is_short"] == False]

print(f"Total videos in target channels: {len(df):,}")
print(f"Shorts: {len(df_shorts):,}")
print(f"Long videos: {len(df_longs):,}")

Total videos in target channels: 12,829
Shorts: 4,356
Long videos: 8,473


In [ ]:
# Calculate channel-level statistics and comparisons
stats = []
for i, channel in enumerate(df["channelTitle"].unique()):
    s_count = len(df_shorts[df_shorts["channelTitle"] == channel])
    l_count = len(df_longs[df_longs["channelTitle"] == channel])
    comparisons = s_count * l_count
    stats.append({
        "Channel": channel,
        "Shorts": s_count,
        "Longs": l_count,
        "Comparisons": comparisons
    })

stats_df = pd.DataFrame(stats)
total_comparisons = stats_df["Comparisons"].sum()
print("\nChannel Statistics:")
display(stats_df)
print(f"Total comparisons required: {total_comparisons:,}")


Channel Statistics:


,Channel,Shorts,Longs,Comparisons
0,Camihawke,16,17,272
1,Gli Autogol,93,633,58869
2,ChiaraMaciTv,2,198,396
3,Canale di Venti,70,600,42000
4,Cotto al Dente,263,769,202247
5,Raissa & Momo,49,39,1911
6,roccotnl,63,21,1323
7,The Lady,100,882,88200
8,Valentina Barbieri,212,2,424
9,THOMAS BASILICO,347,10,3470


Total comparisons required: 1,258,808


## Subset Matching Algorithm & Pre-Filter

We need to check if the **entire** Short transcript is a subset of a Long video transcript. 
Instead of doing character-level comparisons, we use **word-level tokenization** which decreases sequence lengths and speeds up matching by **5x-6x**.

### The Word-Set Pre-Filter:
To avoid running the expensive `difflib.SequenceMatcher` on thousands of unrelated video pairs, we check the intersection of unique words between the Short and the Long video:
$$\text{Word Intersection Ratio} = \frac{\text{Unique words in Short } \cap \text{ Unique words in Long}}{\text{Unique words in Short}}$$
If this ratio is less than the threshold (e.g. 0.8), it is mathematically impossible for the Short transcript to be an 80%+ subset of the Long video transcript. We can **instantly skip** these comparisons.

In [ ]:
def get_words(text):
    if not isinstance(text, str):
        return []
    clean = text.lower().replace("\n", " ").replace("\r", " ")
    return clean.split()

def title_overlap_ratio(short_title, long_title):
    """Fraction of short title words found anywhere in the long title."""
    s_words = set(get_words(short_title))
    l_words = set(get_words(long_title))
    if not s_words:
        return 0.0
    return len(s_words & l_words) / len(s_words)

def find_subset_matches(shorts_df, longs_df, threshold=0.8, title_threshold=0.5):
    shorts_data = []
    for _, row in shorts_df.iterrows():
        words = get_words(row["local_transcript"])
        if words:
            shorts_data.append({
                "videoId": row["videoId"],
                "title": row.get("title", ""),
                "words": words,
                "set": set(words)
            })

    longs_data = []
    for _, row in longs_df.iterrows():
        words = get_words(row["local_transcript"])
        if words:
            longs_data.append({
                "videoId": row["videoId"],
                "title": row.get("title", ""),
                "words": words,
                "set": set(words)
            })

    matches = []
    comparisons_run = 0
    comparisons_skipped = 0

    start_time = time.time()

    for s in shorts_data:
        best_ratio = 0.0
        best_long = None

        for l in longs_data:
            comparisons_run += 1

            # 1. Fast word-set pre-filter
            intersection = s["set"].intersection(l["set"])
            if len(intersection) / len(s["set"]) < threshold:
                comparisons_skipped += 1
                continue

            # 2. Find the single longest contiguous word run shared between short and long.
            #    If that one block covers >= threshold of the short, the short is a clip of the long.
            #    Scattered small matches cannot accumulate into a false positive here.
            match = difflib.SequenceMatcher(None, s["words"], l["words"]).find_longest_match(
                0, len(s["words"]), 0, len(l["words"])
            )
            ratio = match.size / len(s["words"])

            if ratio > best_ratio:
                best_ratio = ratio
                best_long = l

        if best_ratio >= threshold and best_long is not None:
            t_ratio = title_overlap_ratio(s["title"], best_long["title"])
            category = "embedded_short" if t_ratio >= title_threshold else "transcript_only"
            matches.append({
                "short_id": s["videoId"],
                "short_title": s["title"],
                "long_id": best_long["videoId"],
                "long_title": best_long["title"],
                "transcript_ratio": round(best_ratio, 4),
                "title_ratio": round(t_ratio, 4),
                "category": category
            })

    duration = time.time() - start_time
    return pd.DataFrame(matches), duration, comparisons_run, comparisons_skipped

In [ ]:
sample_channel = "Camihawke"
print(f"Running test matching on '{sample_channel}'...")

c_shorts = df_shorts[df_shorts["channelTitle"] == sample_channel]
c_longs = df_longs[df_longs["channelTitle"] == sample_channel]

matches_df, duration, run, skipped = find_subset_matches(c_shorts, c_longs, threshold=0.8, title_threshold=0.5)

print(f"\nResults for {sample_channel}:")
print(f"- Time taken: {duration:.4f} seconds")
print(f"- Total possible comparisons: {run}")
print(f"- Skipped by pre-filter: {skipped} ({skipped/run*100:.1f}% reduction in SequenceMatcher workload)")
print(f"- Actual SequenceMatcher evaluations: {run - skipped}")
print(f"- Deduplicated Shorts found: {len(matches_df)} out of {len(c_shorts)} shorts")

if len(matches_df) > 0:
    print(f"\nCategory breakdown:")
    print(matches_df["category"].value_counts().to_string())
    print()
    display(matches_df[["short_title", "long_title", "transcript_ratio", "title_ratio", "category"]])

In [ ]:
SAVE_PATH = "Data_Cleaned/shorts_dedup_matches.jsonl"

if len(matches_df) > 0:
    matches_df.to_json(SAVE_PATH, orient="records", lines=True, force_ascii=False)
    print(f"Saved {len(matches_df)} matches to {SAVE_PATH}")
else:
    print("No matches to save.")

In [ ]:
# Count total shorts and find unmatched shorts
total_shorts = len(c_shorts)
matched_short_ids = set(matches_df["short_id"].values)
unmatched_shorts = c_shorts[~c_shorts["videoId"].isin(matched_short_ids)]

print(f"Total shorts for {sample_channel}: {total_shorts}")
print(f"Matched shorts: {len(matches_df)}")
print(f"Unmatched shorts: {len(unmatched_shorts)}")
print(f"\nUnmatched short IDs:")
for vid in unmatched_shorts["videoId"].values:
    print(f"  - {vid}")

Total shorts for Camihawke: 16


In [ ]:
# Extrapolate run time to total dataset
time_per_comparison = duration / run
extrapolated_total_time = total_comparisons * time_per_comparison

print("\n=== Extrapolation Analysis ===")
print(f"Average time per comparison: {time_per_comparison * 1000: .4f} ms")
print(f"Total Comparisons for 17 channels: {total_comparisons:,}")
print(f"Estimated total execution time: {extrapolated_total_time:.2f} seconds ({extrapolated_total_time/60:.2f} minutes)")


=== Extrapolation Analysis ===
Average time per comparison:  0.2302 ms
Total Comparisons for 17 channels: 472,518
Estimated total execution time: 108.76 seconds (1.81 minutes)
